# 02 — Player Gameweek Histories

Now that we have per-gameweek data for every player, let's explore:
1. What does the gameweek-level data look like?
2. How do top players' points trend over time?
3. How do actual stats compare to expected stats (xG, xA)?
4. What features look promising for prediction?

In [ ]:
import sys
from pathlib import Path
import json
import pandas as pd
import matplotlib.pyplot as plt

project_root = Path.cwd().parent
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

plt.style.use('ggplot')
pd.set_option('display.max_columns', 30)

## Load bootstrap data for player/team lookups

In [ ]:
BOOTSTRAP_PATH = project_root / 'data' / 'raw' / 'bootstrap_static.json'
HISTORIES_DIR = project_root / 'data' / 'raw' / 'player_histories'

with open(BOOTSTRAP_PATH) as f:
    bootstrap = json.load(f)

players_meta = pd.DataFrame(bootstrap['elements'])
teams = pd.DataFrame(bootstrap['teams'])

pos_map = {1: 'GK', 2: 'DEF', 3: 'MID', 4: 'FWD'}
team_map = dict(zip(teams['id'], teams['name']))

players_meta['position'] = players_meta['element_type'].map(pos_map)
players_meta['team_name'] = players_meta['team'].map(team_map)
players_meta['price'] = players_meta['now_cost'] / 10

print(f'Players in bootstrap: {len(players_meta)}')
print(f'History files: {len(list(HISTORIES_DIR.glob("*.json")))}')

## Build a combined gameweek history DataFrame

Load all player histories into one big DataFrame with player metadata joined in.

In [ ]:
rows = []
for hist_file in HISTORIES_DIR.glob('*.json'):
    pid = int(hist_file.stem.split('_')[1])
    with open(hist_file) as f:
        data = json.load(f)
    for gw in data['history']:
        gw['player_id'] = pid
        rows.append(gw)

gw_df = pd.DataFrame(rows)
print(f'Total gameweek rows: {len(gw_df):,}')
print(f'Unique players with history: {gw_df["player_id"].nunique()}')
print(f'Gameweeks covered: GW{gw_df["round"].min()} to GW{gw_df["round"].max()}')
print(f'\nColumns ({len(gw_df.columns)}):\n{list(gw_df.columns)}')

In [ ]:
# Join player metadata
gw_df = gw_df.merge(
    players_meta[['id', 'web_name', 'position', 'team_name']],
    left_on='player_id', right_on='id', how='left'
)
gw_df['price'] = gw_df['value'] / 10  # per-GW price
gw_df['opponent'] = gw_df['opponent_team'].map(team_map)

# Cast expected stats to float
for col in ['expected_goals', 'expected_assists', 'expected_goal_involvements', 'expected_goals_conceded']:
    gw_df[col] = gw_df[col].astype(float)

gw_df.head(3)

## Points distribution per gameweek

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Overall points distribution
gw_df[gw_df['minutes'] > 0]['total_points'].hist(bins=30, ax=axes[0], edgecolor='black')
axes[0].set_title('Points Distribution (players who played)')
axes[0].set_xlabel('Points')
axes[0].set_ylabel('Frequency')

# Points by position
gw_df[gw_df['minutes'] > 0].boxplot(column='total_points', by='position', ax=axes[1])
axes[1].set_title('Points by Position')
axes[1].set_xlabel('Position')
axes[1].set_ylabel('Points')
plt.suptitle('')

plt.tight_layout()
plt.show()

## Top players: points over time

In [ ]:
# Top 6 by total points
top_players = (
    gw_df.groupby(['player_id', 'web_name'])['total_points']
    .sum().sort_values(ascending=False).head(6)
    .reset_index()
)

fig, ax = plt.subplots(figsize=(14, 6))
for _, row in top_players.iterrows():
    player_gws = gw_df[gw_df['player_id'] == row['player_id']].sort_values('round')
    ax.plot(player_gws['round'], player_gws['total_points'].cumsum(),
            marker='o', markersize=3, label=row['web_name'])

ax.set_xlabel('Gameweek')
ax.set_ylabel('Cumulative Points')
ax.set_title('Cumulative Points — Top 6 Players')
ax.legend()
plt.tight_layout()
plt.show()

## Expected vs Actual: xG vs Goals

In [ ]:
# Aggregate per player: total xG vs total goals (for players with enough minutes)
player_agg = (
    gw_df.groupby(['player_id', 'web_name', 'position'])
    .agg(total_xg=('expected_goals', 'sum'),
         total_goals=('goals_scored', 'sum'),
         total_xa=('expected_assists', 'sum'),
         total_assists=('assists', 'sum'),
         total_minutes=('minutes', 'sum'))
    .reset_index()
    .query('total_minutes >= 450')  # at least 5 full matches
)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# xG vs Goals
for pos, color in [('FWD', 'red'), ('MID', 'blue'), ('DEF', 'green')]:
    subset = player_agg[player_agg['position'] == pos]
    axes[0].scatter(subset['total_xg'], subset['total_goals'], alpha=0.5, label=pos, c=color, s=20)
max_val = max(player_agg['total_xg'].max(), player_agg['total_goals'].max())
axes[0].plot([0, max_val], [0, max_val], 'k--', alpha=0.3, label='xG = Goals')
axes[0].set_xlabel('Expected Goals (xG)')
axes[0].set_ylabel('Actual Goals')
axes[0].set_title('xG vs Actual Goals')
axes[0].legend()

# xA vs Assists
for pos, color in [('FWD', 'red'), ('MID', 'blue'), ('DEF', 'green')]:
    subset = player_agg[player_agg['position'] == pos]
    axes[1].scatter(subset['total_xa'], subset['total_assists'], alpha=0.5, label=pos, c=color, s=20)
max_val = max(player_agg['total_xa'].max(), player_agg['total_assists'].max())
axes[1].plot([0, max_val], [0, max_val], 'k--', alpha=0.3, label='xA = Assists')
axes[1].set_xlabel('Expected Assists (xA)')
axes[1].set_ylabel('Actual Assists')
axes[1].set_title('xA vs Actual Assists')
axes[1].legend()

plt.tight_layout()
plt.show()

## Overperformers and underperformers

Players whose actual goals differ most from xG — these are either clinical finishers or due for regression.

In [ ]:
player_agg['goals_vs_xg'] = player_agg['total_goals'] - player_agg['total_xg']

print('=== Overperformers (most goals above xG) ===')
print(player_agg.nlargest(10, 'goals_vs_xg')[['web_name', 'position', 'total_goals', 'total_xg', 'goals_vs_xg']].to_string(index=False))

print('\n=== Underperformers (most goals below xG) ===')
print(player_agg.nsmallest(10, 'goals_vs_xg')[['web_name', 'position', 'total_goals', 'total_xg', 'goals_vs_xg']].to_string(index=False))

## Home vs Away performance

In [ ]:
home_away = (
    gw_df[gw_df['minutes'] > 0]
    .groupby(['position', 'was_home'])
    .agg(avg_points=('total_points', 'mean'),
         avg_xg=('expected_goals', 'mean'),
         avg_bonus=('bonus', 'mean'),
         count=('total_points', 'count'))
    .round(2)
)
home_away.index = home_away.index.set_levels(['Away', 'Home'], level=1)
print(home_away.to_string())

## Rolling form: 3-gameweek rolling average points

In [ ]:
# Calculate 3-GW rolling average for top players
fig, ax = plt.subplots(figsize=(14, 6))

for _, row in top_players.iterrows():
    player_gws = gw_df[gw_df['player_id'] == row['player_id']].sort_values('round')
    rolling = player_gws['total_points'].rolling(3, min_periods=1).mean()
    ax.plot(player_gws['round'], rolling, marker='o', markersize=3, label=row['web_name'])

ax.set_xlabel('Gameweek')
ax.set_ylabel('3-GW Rolling Avg Points')
ax.set_title('Form Trends — Top 6 Players (3-GW Rolling Average)')
ax.legend()
ax.axhline(y=2, color='gray', linestyle='--', alpha=0.3, label='Baseline (2 pts)')
plt.tight_layout()
plt.show()

## Key stats summary

What features look useful for prediction?

In [ ]:
# Correlation of per-GW features with points
feature_cols = ['minutes', 'goals_scored', 'assists', 'bonus', 'bps',
                'expected_goals', 'expected_assists', 'expected_goal_involvements',
                'expected_goals_conceded', 'was_home', 'value',
                'clean_sheets', 'saves', 'penalties_missed']

played = gw_df[gw_df['minutes'] > 0].copy()
corr = played[feature_cols + ['total_points']].corr()['total_points'].drop('total_points').sort_values(ascending=False)

print('=== Feature Correlation with GW Points ===')
print(corr.round(3).to_string())